In [3]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY)
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/axlee/Desktop/Singhealth/AED-OHCA
/Users/axlee/Desktop/Singhealth/AED-OHCA/datasets


In [5]:
resident_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_density_ratios/final_subzone_population_2010_2021.csv")
resident_df = pd.read_csv(resident_filepath)

resident_df

,subzone_n,pln_area_n,Year,AgeGroup,Estimated_Residents
0,ang mo kio town centre,ang mo kio,2010,0 - 4,1118
1,ang mo kio town centre,ang mo kio,2010,10 - 14,1187
2,ang mo kio town centre,ang mo kio,2010,15 - 19,1141
3,ang mo kio town centre,ang mo kio,2010,20 - 24,1036
4,ang mo kio town centre,ang mo kio,2010,25 - 29,1338
...,...,...,...,...,...
69771,yishun west,yishun,2021,65 - 69,2778
69772,yishun west,yishun,2021,70 - 74,1738
69773,yishun west,yishun,2021,75 - 79,876
69774,yishun west,yishun,2021,80 - 84,628


In [6]:
density_summary = resident_df.groupby(['subzone_n', 'pln_area_n', 'Year'])['Estimated_Residents'].sum().reset_index()
density_summary.rename(columns={'Estimated_Residents': 'total'}, inplace=True)
density_summary

,subzone_n,pln_area_n,Year,total
0,admiralty,sembawang,2010,11412
1,admiralty,sembawang,2011,11744
2,admiralty,sembawang,2012,12104
3,admiralty,sembawang,2013,12438
4,admiralty,sembawang,2014,12777
...,...,...,...,...
4267,yunnan,jurong west,2017,36503
4268,yunnan,jurong west,2018,36205
4269,yunnan,jurong west,2019,35914
4270,yunnan,jurong west,2020,35538


In [7]:
seniors_2010_2014 = ['60 - 64', '65 & Over']
seniors_2015_2021 = ['60 - 64', '65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 & Over']

# Mask for 2010 - 2014
mask_1 = (resident_df['Year'] >= 2010) & \
             (resident_df['Year'] <= 2014) & \
             (resident_df['AgeGroup'].isin(seniors_2010_2014))

# Mask for 2015 - 2021
mask_2 = (resident_df['Year'] >= 2015) & \
            (resident_df['Year'] <= 2021) & \
            (resident_df['AgeGroup'].isin(seniors_2015_2021))

senior_residents_df = resident_df[mask_1 | mask_2].copy()

# Group by subzone, planning area, and year
senior_summary = senior_residents_df.groupby(
    ['subzone_n', 'pln_area_n', 'Year']
)['Estimated_Residents'].sum().reset_index()

# Rename the column for clarity
senior_summary.rename(columns={'Estimated_Residents': 'above_60'}, inplace=True)

# Display result
senior_summary

,subzone_n,pln_area_n,Year,above_60
0,admiralty,sembawang,2010,974
1,admiralty,sembawang,2011,1071
2,admiralty,sembawang,2012,1154
3,admiralty,sembawang,2013,1238
4,admiralty,sembawang,2014,1326
...,...,...,...,...
4267,yunnan,jurong west,2017,5337
4268,yunnan,jurong west,2018,5727
4269,yunnan,jurong west,2019,6127
4270,yunnan,jurong west,2020,6526


In [8]:
final_resident_df = pd.merge(
    density_summary,
    senior_summary,
    on = ["subzone_n", "pln_area_n", "Year"],
    how = 'inner'
)

display(final_resident_df)

,subzone_n,pln_area_n,Year,total,above_60
0,admiralty,sembawang,2010,11412,974
1,admiralty,sembawang,2011,11744,1071
2,admiralty,sembawang,2012,12104,1154
3,admiralty,sembawang,2013,12438,1238
4,admiralty,sembawang,2014,12777,1326
...,...,...,...,...,...
4267,yunnan,jurong west,2017,36503,5337
4268,yunnan,jurong west,2018,36205,5727
4269,yunnan,jurong west,2019,35914,6127
4270,yunnan,jurong west,2020,35538,6526


In [9]:
## calculate above 60 proportion for each subzone
final_resident_df['above_60_proportion'] = (final_resident_df['above_60'] / final_resident_df['total']) * 100

# Replace infinity values if area was 0
final_resident_df['above_60_proportion'] = final_resident_df['above_60_proportion'].replace([np.inf, -np.inf], 0)
final_resident_df["above_60_proportion"] = final_resident_df["above_60_proportion"].fillna(0)
final_resident_df

,subzone_n,pln_area_n,Year,total,above_60,above_60_proportion
0,admiralty,sembawang,2010,11412,974,8.534876
1,admiralty,sembawang,2011,11744,1071,9.119550
2,admiralty,sembawang,2012,12104,1154,9.534038
3,admiralty,sembawang,2013,12438,1238,9.953369
4,admiralty,sembawang,2014,12777,1326,10.378023
...,...,...,...,...,...,...
4267,yunnan,jurong west,2017,36503,5337,14.620716
4268,yunnan,jurong west,2018,36205,5727,15.818257
4269,yunnan,jurong west,2019,35914,6127,17.060199
4270,yunnan,jurong west,2020,35538,6526,18.363442


In [10]:
ethnic_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ethnic_density_ratios/final_subzone_ethnicity_population.csv")
ethnicity_df = pd.read_csv(ethnic_filepath)
ethnicity_df

,subzone_n,pln_area_n,Year,Ethnicity,Estimated_Residents
0,admiralty,sembawang,2010,female_chinese,4135
1,admiralty,sembawang,2010,female_indians,509
2,admiralty,sembawang,2010,female_malays,850
3,admiralty,sembawang,2010,female_others,208
4,admiralty,sembawang,2010,male_chinese,4076
...,...,...,...,...,...
34171,yunnan,jurong west,2021,female_others,322
34172,yunnan,jurong west,2021,male_chinese,11330
34173,yunnan,jurong west,2021,male_indians,2065
34174,yunnan,jurong west,2021,male_malays,4004


In [11]:
pivoted_ethnicity_df = pd.merge(
    final_resident_df,
    ethnicity_df,
    on=['subzone_n', 'pln_area_n', 'Year'],
    how='left'
)

pivoted_ethnicity_df = pivoted_ethnicity_df.pivot(
    index=['subzone_n', 'pln_area_n', 'Year', 'total', 'above_60', 'above_60_proportion'], 
    columns='Ethnicity', 
    values='Estimated_Residents'
).reset_index()

# This removes the "Ethnicity" name from the columns axis
pivoted_ethnicity_df.columns.name = None

# If a specific subzone/year combo was missing an ethnicity, it will be NaN. Fill with 0.
pivoted_ethnicity_df = pivoted_ethnicity_df.fillna(0)


ethnic_cols = ['male_chinese', 'female_chinese', 'male_malays', 'female_malays',
                   'male_indians', 'female_indians', 'male_others', 'female_others']

## calculate ethnicity/gender proportion for each subzone 
for col in ethnic_cols:
    pivoted_ethnicity_df[f'{col}_proportion'] = (pivoted_ethnicity_df[col] / pivoted_ethnicity_df['total']) * 100

# Drop the raw counts now that proportions are saved
pivoted_ethnicity_df.drop(columns=ethnic_cols, inplace=True)

# split the table based on the available landuse years (2014 and 2019)

# Split for 2010 - 2018
df_2010_2018 = pivoted_ethnicity_df[
    (pivoted_ethnicity_df['Year'] >= 2010) & (pivoted_ethnicity_df['Year'] <= 2018)
].copy()

# Split for 2019 - 2021
df_2019_2021 = pivoted_ethnicity_df[
    (pivoted_ethnicity_df['Year'] >= 2019) & (pivoted_ethnicity_df['Year'] <= 2021)
].copy()

# save_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/demographic_data_2010_2013.csv")
# df_2010_2013.to_csv(save_filepath)
# save_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/demographic_data_2014_2018.csv")
# df_2014_2018.to_csv(save_filepath)
# save_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/demographic_data_2019_2021.csv")
# df_2019_2021.to_csv(save_filepath)

pivoted_ethnicity_df

,subzone_n,pln_area_n,Year,total,above_60,above_60_proportion,male_chinese_proportion,female_chinese_proportion,male_malays_proportion,female_malays_proportion,male_indians_proportion,female_indians_proportion,male_others_proportion,female_others_proportion
0,admiralty,sembawang,2010,11412,974,8.534876,35.716789,36.233789,7.404487,7.448300,4.644234,4.460217,2.085524,1.822643
1,admiralty,sembawang,2011,11744,1071,9.119550,35.413828,35.958787,7.654973,7.782698,4.615123,4.461853,2.026567,1.813692
2,admiralty,sembawang,2012,12104,1154,9.534038,35.153668,35.731989,7.873430,8.071712,4.576999,4.477859,1.974554,1.817581
3,admiralty,sembawang,2013,12438,1238,9.953369,34.973468,35.560379,8.104197,8.369513,4.550571,4.478212,1.937611,1.833092
4,admiralty,sembawang,2014,12777,1326,10.378023,34.765594,35.376066,8.327463,8.656179,4.523754,4.476794,1.878375,1.847069
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4267,yunnan,jurong west,2017,36503,5337,14.620716,33.350684,32.452127,11.215517,11.092239,5.687204,4.920144,0.712270,0.904035
4268,yunnan,jurong west,2018,36205,5727,15.818257,33.227455,32.346361,11.274686,11.150394,5.758873,4.996547,0.715371,0.914238
4269,yunnan,jurong west,2019,35914,6127,17.060199,33.118004,32.265969,11.324275,11.204544,5.833380,5.078799,0.715598,0.918862
4270,yunnan,jurong west,2020,35538,6526,18.363442,32.967528,32.109291,11.399066,11.275255,5.906354,5.121279,0.720356,0.928583


In [12]:
# add subzone characteristics to the demographic data
subzone_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_combined_characteristics_2015.csv")
subzone_characteristics_2015 = pd.read_csv(subzone_filepath)
subzone_characteristics_2015['pln_area_n'] = subzone_characteristics_2015['pln_area_n'].str.strip().str.lower()
subzone_characteristics_2015['subzone_n'] = subzone_characteristics_2015['subzone_n'].str.strip().str.lower()

subzone_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_combined_characteristics_2020.csv")
subzone_characteristics_2020 = pd.read_csv(subzone_filepath)
subzone_characteristics_2020['pln_area_n'] = subzone_characteristics_2020['pln_area_n'].str.strip().str.lower()
subzone_characteristics_2020['subzone_n'] = subzone_characteristics_2020['subzone_n'].str.strip().str.lower()


columns_of_interest = ['subzone_n', 'pln_area_n', "pri_classification", "business_1_encoding",
                       "business_2_encoding", "business_park_encoding", "school_encoding", "airport"]

df_2010_2018 = pd.merge(
    df_2010_2018,
    subzone_characteristics_2015[columns_of_interest],
    on=['subzone_n', 'pln_area_n'],
    how='left'
)

# Merge 2020 characteristics with the 2019-2021 demographic group
df_2019_2021 = pd.merge(
    df_2019_2021,
    subzone_characteristics_2020[columns_of_interest],
    on=['subzone_n', 'pln_area_n'],
    how='left'
)

# Handle NaN values for the new characteristic columns
# Landuse encodings should be 0 if no match was found, and classification as 'unknown'
encoding_cols = ['business_1_encoding', 'business_2_encoding', 
                 'business_park_encoding', 'school_encoding', 'airport']

for df in [df_2010_2018, df_2019_2021]:
    df[encoding_cols] = df[encoding_cols].fillna(0).astype(int)
    df['pri_classification'] = df['pri_classification'].fillna('unknown')
    # Fill all other remaining NaN values (e.g., proportions, total counts) with 0
    df.fillna(0, inplace=True)

    # Set to 1 if residential, 0 otherwise
    df['is_residential'] = (df['pri_classification'] == 'residential').astype(int)
    # Drop the pri_classification column
    df.drop(columns=['pri_classification'], inplace=True)

save_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/combined_demographic_geo_2010_2018.csv")
df_2010_2018.to_csv(save_filepath)
save_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/combined_demographic_geo_2019_2021.csv")
df_2019_2021.to_csv(save_filepath)